# consoleBert Colab
This notebook shows how to compute SBERT embeddings for a small sample set, save vectors, and run a nearest-neighbor lookup for retrieval-based transforms.

In [ ]:
# Install required libraries
!pip install -q sentence-transformers scikit-learn

In [ ]:
from sentence_transformers import SentenceTransformer
import json, numpy as np
model = SentenceTransformer('all-MiniLM-L6-v2')

# Load the sample sentences included in the repository (adjust path if needed)
with open('../data/sample_sentences.json','r',encoding='utf-8') as f:
    samples = json.load(f)
texts = [s['text'] for s in samples]

# Compute embeddings and save them
emb = model.encode(texts, convert_to_numpy=True)
np.save('../data/embeddings.npy', emb)
with open('../data/texts.json','w',encoding='utf-8') as f:
    json.dump(texts, f, ensure_ascii=False, indent=2)

print('Saved', len(texts), 'texts and embeddings')

In [ ]:
# Build a simple nearest-neighbor index with scikit-learn
from sklearn.neighbors import NearestNeighbors
emb = np.load('../data/embeddings.npy')
nbr = NearestNeighbors(n_neighbors=1, metric='cosine').fit(emb)

def find_nearest(query):
    v = model.encode([query])
    d, idx = nbr.kneighbors(v, return_distance=True)
    return idx[0][0], float(d[0][0])

print(find_nearest('Please send the draft by Friday.'))

### Next steps
- Expand the sample sentence pool to cover more tone/X/Y cells.
- Use the saved embeddings and index in the FastAPI `/transform` endpoint for retrieval.
- Optionally fine-tune a small seq2seq model if retrieval quality needs to improve.